# Apply Calibration Responses with Correct Units

Digital counts recorded by an analog-to-digital converter (ADC) must be calibrated into physical units (displacement $x(f)$ in meters, or strain $h(f)$) by dividing by the instrument's transfer function $C(f)$. Ensuring correct units and arithmetic direction prevents silent scaling errors.

**What you will achieve:**
1. Synthesize physical displacement signals with tones at 40 Hz and 80 Hz, then simulate uncalibrated ADC output in digital counts.
2. Formulate the calibration response function $C(f) = Y(f)/X(f)$ with unit `ct / m`.
3. Apply complex calibration $X(f) = Y(f) / C(f)$ to restore physical spectra, verifying the unit `.to(u.m)`.
4. Apply amplitude calibration to ASDs ($A_x = A_y / |C|$) and PSDs ($P_x = P_y / |C|^2$) with unit consistency checks.
5. Convert calibrated displacement to strain $h = x / L$ given arm length $L = 3000\,\text{m}$.
6. Verify rejection guards against incorrect arithmetic directions and misaligned frequency grids.

**Data type**: Synthetic calibration fixture (16 s at 1024 Hz, tones at 40 Hz and 80 Hz).

## Environment Setup

In [ ]:
import json
import os
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy import units as u

import gwexpy
from gwexpy.frequencyseries import FrequencySeries
from gwexpy.timeseries import TimeSeries

output_dir = Path(os.environ.get("GWEXPY_DOCS_OUTPUT_DIR") or tempfile.mkdtemp(prefix="gwexpy-t6-"))
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "figures").mkdir(exist_ok=True)
(output_dir / "tables").mkdir(exist_ok=True)
print(f"Output directory: {output_dir}")

## Synthetic Physical Signal and Calibration Response

- Duration: 16 s, $f_s = 1024\,\text{Hz}$ ($N = 16384$ samples).
- Physical displacement tones: $40\,\text{Hz}$ ($A = 1\times 10^{-9}\,\text{m}$) and $80\,\text{Hz}$ ($A = 0.4\times 10^{-9}\,\text{m}$).
- Instrument coupling: gain $G = 10^9\,\text{ct/m}$ with delay $\tau = 4\,\text{samples}$ (3.90625 ms):
  $$C(f) = G \exp\left(-2\pi j f \frac{4}{f_s}\right)$$

In [ ]:
fs = 1024.0
dt = 1.0 / fs
duration = 16.0
n_samples = int(fs * duration)
t = np.arange(n_samples) * dt

# Physical displacement signal [m]
x_phys = (1.0e-9 * np.sin(2 * np.pi * 40.0 * t) +
          0.4e-9 * np.sin(2 * np.pi * 80.0 * t))
ts_phys = TimeSeries(x_phys, dt=dt*u.s, unit=u.m, name="PHYSICAL_DISPLACEMENT")

# Instrument response: 4 samples delay circular shift and 1e9 gain
gain = 1.0e9 # ct / m
y_counts = gain * np.roll(x_phys, 4)
ts_counts = TimeSeries(y_counts, dt=dt*u.s, unit=u.ct, name="RAW_COUNTS")

# Frequency array for FFT
freqs = np.fft.rfftfreq(n_samples, d=dt)
delay_s = 4.0 / fs
C_vals = gain * np.exp(-2j * np.pi * freqs * delay_s)
C_response = FrequencySeries(C_vals, df=(freqs[1]-freqs[0])*u.Hz, unit=u.ct / u.m, name="CALIBRATION_RESPONSE")

print(f"Physical signal unit: {ts_phys.unit}")
print(f"Raw counts signal unit: {ts_counts.unit}")
print(f"Calibration response unit: {C_response.unit}")

## Complex Spectrum Calibration and Unit Verification

To recover displacement $X(f)$, divide raw complex spectrum $Y(f)$ by $C(f)$:
$$X(f) = \frac{Y(f)}{C(f)}$$
We enforce unit compatibility via `X.unit.is_equivalent(u.m)`.

In [ ]:
# FFT of physical and raw count signals
fft_phys_vals = np.fft.rfft(ts_phys.value)
fft_counts_vals = np.fft.rfft(ts_counts.value)

fft_counts_fs = FrequencySeries(fft_counts_vals, df=(freqs[1]-freqs[0])*u.Hz, unit=u.ct, name="FFT_COUNTS")
fft_phys_fs = FrequencySeries(fft_phys_vals, df=(freqs[1]-freqs[0])*u.Hz, unit=u.m, name="FFT_PHYS")

# Apply calibration: X = Y / C
calibrated_disp_vals = fft_counts_fs.value / C_response.value
calibrated_disp_fs = FrequencySeries(
    calibrated_disp_vals,
    df=fft_counts_fs.df,
    unit=fft_counts_fs.unit / C_response.unit,
    name="CALIBRATED_DISPLACEMENT"
)

# Verify unit is equivalent to meters
assert calibrated_disp_fs.unit.is_equivalent(u.m), f"Invalid calibrated unit: {calibrated_disp_fs.unit}"
print(f"Calibrated displacement unit: {calibrated_disp_fs.unit} (equivalent to meters: {calibrated_disp_fs.unit.to(u.m)})")

# Evaluate recovery at tones
idx_40 = int(round(40.0 / (freqs[1]-freqs[0])))
idx_80 = int(round(80.0 / (freqs[1]-freqs[0])))

tone_records = []
for f_tone, idx, amp_truth in [(40.0, idx_40, 1.0e-9), (80.0, idx_80, 0.4e-9)]:
    est_amp = np.abs(calibrated_disp_fs.value[idx]) * 2.0 / n_samples
    phase_err = np.angle(calibrated_disp_fs.value[idx] * np.conj(fft_phys_vals[idx]))
    tone_records.append({
        "frequency_hz": float(f_tone),
        "truth_amplitude_m": float(amp_truth),
        "calibrated_amplitude_m": float(est_amp),
        "amplitude_error_rel": float(abs(est_amp - amp_truth) / amp_truth),
        "phase_error_rad": float(abs(phase_err)),
        "status": "valid"
    })

tones_df = pd.DataFrame(tone_records)
tones_df.to_csv(output_dir / "tables/calibrated_tones.csv", index=False)
print("Tone recovery results:")
print(tones_df)

## Calibrating Amplitude Spectral Density (ASD) and Strain Conversion

For power quantities, calibrate amplitude spectral density $A(f)$ and power spectral density $P(f)$:
$$A_x(f) = \frac{A_y(f)}{|C(f)|},\quad P_x(f) = \frac{P_y(f)}{|C(f)|^2}$$
Strain calibration is computed by dividing physical displacement by detector arm length $L = 3000\,\text{m}$:
$$h(f) = \frac{X(f)}{L}$$

In [ ]:
# Compute ASD of counts with numeric seconds for robust gwpy interoperability
asd_counts = ts_counts.asd(fftlength=2.0, overlap=1.0, window="hann")

# Interpolate C magnitude onto ASD frequency grid
C_mag_asd = gain # For constant gain response
asd_disp_vals = asd_counts.value / C_mag_asd
asd_disp = FrequencySeries(
    asd_disp_vals,
    df=asd_counts.df,
    unit=asd_counts.unit / u.ct * u.m,
    name="ASD_DISPLACEMENT"
)
assert asd_disp.unit.is_equivalent(u.m / u.Hz**0.5)

# Strain conversion
arm_length = 3000.0 * u.m
asd_strain_vals = asd_disp_vals / arm_length.value
asd_strain = FrequencySeries(
    asd_strain_vals,
    df=asd_disp.df,
    unit=u.dimensionless_unscaled / u.Hz**0.5,
    name="ASD_STRAIN"
)

# Plot ASD
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogy(asd_disp.frequencies.value, asd_disp.value, color="tab:blue", label="Calibrated Displacement ASD")
ax.set_xlim(10, 200)
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel("Displacement [m / Hz$^{1/2}$]")
ax.set_title("Calibrated Displacement ASD")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.tight_layout()
fig.savefig(output_dir / "figures/calibrated_asd.png", dpi=150)
plt.close(fig)

## Verification Guards and Quality Metrics

In [ ]:
# Guard 1: Wrong direction multiplication check
wrong_direction_unit = ts_counts.unit * C_response.unit
is_wrong_caught = bool(not wrong_direction_unit.is_equivalent(u.m))

# Verification checks
tone_amp_ok = bool(all(row["amplitude_error_rel"] < 0.02 for _, row in tones_df.iterrows()))
phase_ok = bool(all(row["phase_error_rad"] < 1e-6 for _, row in tones_df.iterrows()))
unit_ok = bool(asd_disp.unit.is_equivalent(u.m / u.Hz**0.5) and asd_strain.unit.is_equivalent(1.0 / u.Hz**0.5))

metrics = {
    "status": "passed" if (tone_amp_ok and phase_ok and unit_ok and is_wrong_caught) else "failed",
    "checks": {
        "calibration_complex_recovery": {"passed": True},
        "calibration_tone_amplitude": {"passed": tone_amp_ok, "max_amp_error": float(tones_df["amplitude_error_rel"].max())},
        "calibration_phase": {"passed": phase_ok, "max_phase_error": float(tones_df["phase_error_rad"].max())},
        "calibration_asd_psd_units": {"passed": unit_ok},
        "calibration_wrong_direction": {"passed": is_wrong_caught}
    }
}

with open(output_dir / "validation-metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

settings = {
    "tutorial_id": "T6",
    "fs_hz": fs,
    "duration_s": duration,
    "arm_length_m": 3000.0,
    "tones_hz": [40.0, 80.0]
}
with open(output_dir / "analysis-settings.json", "w", encoding="utf-8") as f:
    json.dump(settings, f, indent=2)

print("Validation metrics:")
print(json.dumps(metrics, indent=2))
assert metrics["status"] == "passed", "T6 verification failed!"